# Stage 3 — Regularizer experiment (Set5 ×4, loss × regularizer), tracked in W&B

**Codebase:** `lakshjain7/deep-image-prior` (the repo that produced Stage 1/2: `super_resolution_cli.py`, same Downsampler,
same data pipeline). The paper's regularizers are **ported verbatim** from `capo-urjc/dip-sisr@e2c80f0` into
`utils/regularizers.py`; a unit test compares them numerically with the official file (cell 5).
The official `main.py` is *not* run: it hard-codes `CUDA_VISIBLE_DEVICES="3"`, saves a checkpoint every iteration, and its `datasets/` folder
(no `__init__.py`) is shadowed by the Hugging Face `datasets` package that Colab ships — the likely cause of the earlier `ModuleNotFoundError`.

**Matrix:** 5 images × 3 losses (L1, L1-EM, M-L1) × 5 conditions (none, TV, NN, TVSSAH, BD) × 1 seed = **75 runs**, one W&B run each.

**Labelling:** with the default `PROTOCOL = "adapted"` this is an *adapted* experiment (lr 0.01, noise 0.03, 2000 it — the Stage 1 settings), **not** an exact paper reproduction.
Regularizer weights are the paper's Table 3 values for Set5 (TV 0.5, NN 6, TVSSAH 1, BD 0.3; α=2.8, β=0.35), tuned by the authors for their own protocol (lr 0.001, noise 0.01, 5500 it, bicubic LR), so under the adapted protocol they are paper values, not re-tuned ones. Every run records this in its W&B config.

**Resume:** just re-run the *Matrix* cell. A configuration is skipped only if W&B holds a run for it with `state == finished` **and** `summary.status == "complete"`.

## 1. Setup — repo, dependencies

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

REPO_URL = "https://github.com/lakshjain7/deep-image-prior.git"
BRANCH = "claude/epic-ride-rnqbk4"          # change to "master" once this branch is merged
REPO_DIR = Path(os.environ.get("DIP_REPO_DIR", "/content/deep-image-prior"))

if "DIP_REPO_DIR" not in os.environ:          # normal Colab path
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{BRANCH}"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))

for mod, pkg in [("wandb", "wandb"), ("skimage", "scikit-image"), ("cv2", "opencv-python-headless"), ("pytest", "pytest")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import torch, wandb
print("repo commit :", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| wandb", wandb.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available() or os.environ.get("DIP_ALLOW_CPU"), "No GPU: Runtime > Change runtime type > GPU"

## 2. Weights & Biases login (no key is typed into the notebook or chat)

Preferred: Colab **Secrets** (key icon in the left bar) → add a secret named `WANDB_API_KEY` (value from https://wandb.ai/authorize) and enable *Notebook access*.
If no secret exists, `wandb.login()` shows a masked prompt instead.

In [ ]:
if os.environ.get("WANDB_MODE") != "offline":
    try:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print("No usable Colab secret (", type(e).__name__, ") -> interactive wandb.login()")
    wandb.login()
    print("W&B user/entity:", wandb.Api().default_entity)

## 3. Experiment configuration

In [ ]:
import stage3
from stage3 import IMAGES, LOSSES, REGULARIZERS

ENTITY   = None                       # None = your default W&B entity; or set a team name
PROJECT  = "dip-sr-regularizers"
PROTOCOL = "adapted"                  # "adapted" (Stage 1 settings) or "paper" (lr 0.001, noise 0.01, 5500 it)
SEEDS    = [7]                        # Stage 1 used 7 and 123; extend to [7, 123] only if compute allows
LR_METHOD = "pil-lanczos"             # how LR is made: "pil-lanczos" (Stages 1-3 so far) or "bicubic" (official dip-sisr code)
RUN_IMAGES, RUN_LOSSES, RUN_REGS = IMAGES, LOSSES, REGULARIZERS   # edit to run a subset, e.g. RUN_IMAGES=["butterfly"]
REG_WEIGHTS = dict(stage3.DEFAULT_REG_WEIGHTS)   # edit here if you decide on other weights
TIME_BUDGET_MIN = 150                 # stop launching new runs after this many minutes in one session

P = stage3.PROTOCOLS[PROTOCOL]
CFG = {"factor": 4, "lr": P["lr"], "reg_noise_std": P["reg_noise_std"],
       "num_iter": int(os.environ.get("DIP_ITERS", P["num_iter"])), "log_freq": 100,
       "protocol_note": P["note"], "tags": [PROTOCOL, LR_METHOD, "regularizer-study"]}
OUT = Path(os.environ.get("DIP_OUT", "/content/dip_stage3_outputs")); (OUT / "logs").mkdir(parents=True, exist_ok=True)

jobs = stage3.build_jobs(RUN_IMAGES, RUN_LOSSES, RUN_REGS, SEEDS, PROTOCOL, REG_WEIGHTS, LR_METHOD)
tracker = stage3.Tracker(ENTITY, PROJECT, OUT / "runs")
print(f"{len(jobs)} planned configurations | {PROTOCOL}: {CFG['num_iter']} it, lr {CFG['lr']}, noise {CFG['reg_noise_std']} | LR: {LR_METHOD}")
print("Regularizer weights:")
for r in REGULARIZERS: print(f"  {r:7s} {REG_WEIGHTS[r]:<6g} {stage3.WEIGHT_PROVENANCE[r]}")

## 4. Set5 data (HR images; verified by name and size, never guessed)

In [ ]:
import urllib.request, tarfile, zipfile
SET5_DIR = Path(os.environ.get("DIP_SET5_DIR", "/content/Set5"))

def try_download():
    """Candidate sources are unverified guesses; find_set5() validates whatever arrives."""
    SET5_DIR.mkdir(parents=True, exist_ok=True)
    for url in ["https://huggingface.co/datasets/eugenesiow/Set5/resolve/main/data/Set5_HR.tar.gz"]:
        try:
            tmp = SET5_DIR / Path(url).name
            urllib.request.urlretrieve(url, tmp)
            (tarfile.open(tmp) if tmp.suffix == ".gz" else zipfile.ZipFile(tmp)).extractall(SET5_DIR)
            return True
        except Exception as e:
            print("download failed:", url, type(e).__name__, e)
    return False

try:
    image_paths = stage3.find_set5(SET5_DIR)
except FileNotFoundError:
    try_download()
    image_paths = stage3.find_set5(SET5_DIR)   # raises with a clear message if still wrong
for k, v in image_paths.items(): print(f"{k:10s} {v}")

If this cell raises, upload the five **HR** Set5 images (the ones used in Stage 1) to `/content/Set5/` (any sub-folder) and re-run it. Do not continue with LR images.

## 5. Verify the regularizer implementation against the official code

Clones `capo-urjc/dip-sisr` at the pinned commit **only to read `losses/functions.py` by file path** (never put it on `sys.path`) and checks that
`utils/regularizers.py` reproduces TV, nuclear norm, SSAHTV and the Schiavi/BD term **bit-for-bit**.

In [ ]:
OFFICIAL_COMMIT = "e2c80f0e60880a868e77dd024b70cb4a2e4b3ed5"
ref = Path(os.environ.get("DIP_SISR_DIR", "/content/dip-sisr-ref"))
if not ref.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/capo-urjc/dip-sisr.git", str(ref)], check=True)
    subprocess.run(["git", "-C", str(ref), "checkout", "-q", OFFICIAL_COMMIT], check=True)
env = dict(os.environ, DIP_SISR_DIR=str(ref))
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_regularizers.py"], env=env, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-400:])
assert r.returncode == 0, "Regularizer implementation does not match the official code - stop."

## 6. Smoke test — one real regularised configuration (200 it)

Recorded in W&B as a **validation run** (`job_type=smoke`, tag `smoke`, config `run_key` prefixed `smoke__`), so it never enters the main results.
The second call must be skipped, proving the resume check works.

In [ ]:
import copy, json
from IPython.display import Image as IPImage, display

sj = copy.deepcopy(next(j for j in jobs if j["reg"] != "none"))   # first regularised job of the current selection
sj["key"] = "smoke__" + sj["key"]
SCFG = dict(CFG, num_iter=int(os.environ.get("DIP_SMOKE_ITERS", 200)), job_type="smoke",
            tags=["smoke", "validation"], name_prefix="smoke")

if tracker.is_complete(sj["key"]):
    print("smoke run already complete in W&B/local -> not repeating")
else:
    rec = stage3.run_job(sj, SCFG, image_paths[sj["image"]], tracker, OUT / "logs")
    print(rec.get("status"), {k: rec[k] for k in ("returncode", "elapsed_s") if k in rec})
    assert rec["status"] == "complete", rec.get("error_tail")

m = json.loads((tracker.run_dir(sj["key"]) / "metrics.json").read_text())
print({k: round(m[k], 4) for k in ("psnr_hr", "ssim_hr", "psnr_y_border4", "last_iter_psnr_hr", "psnr_bicubic_baseline", "final_fidelity_loss", "final_reg_loss")})
assert m["status"] == "complete" and m["config"]["regularizer"] == sj["reg"] and m["config"]["loss"] == sj["loss"] and m["final_reg_loss"] > 0
print("W&B records for this key:", tracker.wandb_runs(sj["key"]))
print("resume check ->", stage3.run_job(sj, SCFG, image_paths[sj["image"]], tracker, OUT / "logs")["status"])   # expect skipped_complete
display(IPImage(str(next(tracker.run_dir(sj["key"]).glob("final_sr_x4.png")))))

Open the run in W&B and check: config (loss, regularizer, weight, seed, lr, noise, git commit), curves (`loss`, `fidelity_loss`, `reg_loss`, `psnr_HR`), the `comparison` image, and the `result-…` artifact. **Run the matrix only after this looks right.**

## 7. Run the matrix (resumable)

One W&B run per configuration (`job_type=stage3-matrix`, group `stage3-<protocol>`, name `s3_<image>_<loss>_<reg>_seed<seed>`, tags for loss/regularizer/image/seed).
Unregularised baselines run first. A failed process makes the CLI mark its W&B run **failed** (exit code 1, `summary.error`); the config is retried next time as a new attempt, and the failed run stays visible.
After a disconnect: re-run cells 1–4 and this cell (cells 5–6 may be skipped).

In [ ]:
import json, time, pandas as pd
t_start, done_now, durations, failures = time.time(), 0, [], []
for n, job in enumerate(jobs, 1):
    spent = (time.time() - t_start) / 60
    est = (sum(durations) / len(durations) / 60) if durations else 0
    if spent + 1.2 * est > TIME_BUDGET_MIN:
        print(f"Time budget reached ({spent:.0f} min). Re-run this cell later to continue."); break
    rec = stage3.run_job(job, CFG, image_paths[job["image"]], tracker, OUT / "logs")
    if rec["status"] == "skipped_complete":
        print(f"[{n}/{len(jobs)}] skip   {job['key']}"); continue
    durations.append(rec["elapsed_s"]); done_now += rec["status"] == "complete"
    print(f"[{n}/{len(jobs)}] {rec['status']:8s} {job['key']}  ({rec['elapsed_s']/60:.1f} min)")
    if rec["status"] != "complete":
        failures.append(rec); print("   ", rec.get("error_tail", "")[-300:].replace("\n", "\n    "))
        with open(OUT / "failures.jsonl", "a") as f: f.write(json.dumps(rec) + "\n")
print(f"This session: {done_now} completed, {len(failures)} failed.")

## 8. Results table, failure log and summary (read from W&B)

Builds the consolidated table from W&B (so it works after a disconnect), saves CSVs, and logs a `stage3-summary` run with the tables and heat-map.
Primary metric: `psnr_hr` = RGB PSNR of the final reconstruction (clean input) vs. the cropped HR image. Also reported: SSIM, Y-channel PSNR (4 px border, `evaluate_set5.py` protocol) and `last_iter_psnr_hr` (the Stage 1 "last-recorded" style value).
Δ = metric minus the unregularised run with the same image, loss and seed.

In [ ]:
import numpy as np, matplotlib; import matplotlib.pyplot as plt
from IPython.display import display
df = stage3.collect_results(jobs, tracker)
S = stage3.summarize(df, "psnr_hr")
res = S["frame"]; res.to_csv(OUT / "results.csv", index=False)
bad = res[res.status != "complete"]; bad.to_csv(OUT / "failures.csv", index=False)
print(f"complete {int((res.status=='complete').sum())}/{len(res)} | not complete {len(bad)}")
if len(bad): display(bad[["key", "status"] + [c for c in ("error", "url") if c in bad]])

print("\nUnregularised baseline PSNR (dB)"); display(S["baseline"].round(3))
print("Mean ΔPSNR vs. baseline (dB), by loss × regularizer"); display(S["mean_delta"].round(3))
print("Images improved (count) / runs available"); display(S["images_improved"].astype("Int64").astype(str) + " / " + S["n_complete"].astype("Int64").astype(str))
print("Per-image ΔPSNR (dB)"); display(S["per_image_delta"].round(3))

# Informational sanity check against Stage 1 (reported means over seeds 7 AND 123, metric definition unverified)
STAGE1_REPORTED = {"l1": 40.5397, "l1-em": 39.5393, "m-l1": 39.2585}
chk = res[(res.reg == "none") & (res.status == "complete")].groupby("loss")[["last_iter_psnr_hr", "psnr_hr"]].mean()
chk["stage1_reported_mean"] = pd.Series(STAGE1_REPORTED); display(chk.round(3))

fig, ax = plt.subplots(figsize=(6, 3)); md_ = S["mean_delta"]
if md_.size:
    im = ax.imshow(md_.values, cmap="RdBu", vmin=-abs(md_.values).max(), vmax=abs(md_.values).max())
    ax.set_xticks(range(md_.shape[1]), md_.columns); ax.set_yticks(range(md_.shape[0]), md_.index)
    for (i, j), v in np.ndenumerate(md_.values): ax.text(j, i, f"{v:+.2f}", ha="center", va="center")
    ax.set_title("Mean ΔPSNR (dB) vs. no regularizer"); fig.colorbar(im); fig.tight_layout(); fig.savefig(OUT / "mean_delta_heatmap.png", dpi=150)
plt.show()

with wandb.init(entity=tracker.entity, project=PROJECT, job_type="stage3-summary", name=f"s3_summary_{PROTOCOL}_{LR_METHOD}_{len(jobs)}runs", tags=["summary", "stage3", PROTOCOL],
                config={"protocol": PROTOCOL, "lr_method": LR_METHOD, "seeds": SEEDS, "reg_weights": REG_WEIGHTS, "planned": len(res), "complete": int((res.status == "complete").sum())}) as sr:
    sr.log({"results": wandb.Table(dataframe=res.drop(columns=[c for c in ("error",) if c in res]).astype(str)),
            "mean_delta_psnr": wandb.Table(dataframe=S["mean_delta"].reset_index()),
            "failures": wandb.Table(dataframe=bad.astype(str)), **({"heatmap": wandb.Image(str(OUT / "mean_delta_heatmap.png"))} if md_.size else {})})
print("CSV files:", OUT / "results.csv", OUT / "failures.csv")

## 9. (Optional) Weight calibration — off by default

The weights come from the paper (Table 3), but were tuned for the paper's protocol. If you want a small sweep, enable this on **one** image/loss. Caveat: choosing weights by ground-truth PSNR on an image that is also in the final matrix biases that image's result; report it as tuned.
Runs are tagged `calibration` (`job_type=stage3-calibration`) and are never part of the matrix table.

In [ ]:
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    for reg in ["tv", "nn", "bd"]:
        for mult in [0.2, 1.0, 5.0]:
            w = REG_WEIGHTS[reg] * mult
            cj = {"image": "baby", "loss": "l1", "reg": reg, "weight": w, "seed": SEEDS[0], "protocol": PROTOCOL, "lr_method": LR_METHOD,
                  "key": "calib__" + stage3.run_key("baby", "l1", reg, SEEDS[0], PROTOCOL, w, LR_METHOD)}
            print(cj["key"], stage3.run_job(cj, dict(CFG, job_type="stage3-calibration", tags=["calibration"], name_prefix="calib"),
                                            image_paths[cj["image"]], tracker, OUT / "logs")["status"])